# Figures Generator
Generates all figures for `Justin.tex`. Run this notebook to regenerate figures in `M2R_Justin_Part/figures/`.

In [ ]:
import sys
import os

# Add project src to path so we can import the shared models
PROJECT_ROOT = os.path.abspath(os.path.join('..', '..', '..', '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

FIGURES_DIR = os.path.join('M2R_Justin_Part', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print('Project root:', PROJECT_ROOT)
print('Figures dir: ', FIGURES_DIR)

## Figure: Monte Carlo Convergence
Shows how the mean absolute error $|\hat{C} - C_{BS}|$ decays as the number of simulation paths $M$ increases, compared to the theoretical $\mathcal{O}(M^{-1/2})$ rate.

In [ ]:
from pricing.models import mc_european_gbm, bsm_european

# Parameters
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2
bs_price = bsm_european(S, K, T, r, sigma)
print(f'Black-Scholes price: {bs_price:.6f}')

# Log-spaced M values from 10 to 20000
M_values = np.unique(np.round(np.logspace(1, np.log10(20000), 55)).astype(int))
n_trials = 50  # independent trials per M for a stable error estimate

mean_errors = []
for M in M_values:
    errs = [
        abs(mc_european_gbm(S, K, T, r, sigma, int(M), seed=trial)[0] - bs_price)
        for trial in range(n_trials)
    ]
    mean_errors.append(np.mean(errs))

mean_errors = np.array(mean_errors)

# Scale theoretical O(1/sqrt(M)) line to data
scale = mean_errors[5] * np.sqrt(M_values[5])
theory = scale / np.sqrt(M_values)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.loglog(M_values, mean_errors,
          color='#2563EB', lw=1.8, marker='o', markersize=3.5,
          label=r'Mean abs. error $|\hat{C} - C_{\mathrm{BS}}|$')
ax.loglog(M_values, theory,
          color='#DC2626', lw=1.5, ls='--',
          label=r'Reference: $\mathcal{O}(M^{-1/2})$')

ax.set_xlabel('Number of simulation paths $M$', fontsize=11)
ax.set_ylabel('Mean absolute error', fontsize=11)
ax.set_title('Monte Carlo Convergence under Black-Scholes', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', ls=':', alpha=0.5)
ax.set_xlim([M_values[0], M_values[-1]])

plt.tight_layout()

out = os.path.join(FIGURES_DIR, 'mc_convergence_plot.png')
plt.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')